# Fine-tune your German tutor model (free Colab GPU)

**Before running:** Runtime menu -> Change runtime type -> T4 GPU (free tier).

Steps: install libs -> load base model -> load your dataset.jsonl -> train -> save/export.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps "git+https://github.com/unslothai/unsloth.git"

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## Upload your dataset
Upload `dataset.jsonl` (built with `prepare_dataset.py`) using the Colab file browser (left sidebar), or run the cell below to upload directly.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select dataset.jsonl

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(formatting_func)
print(dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## Save the model
Two options: save LoRA adapters only (small, fast) or merge + export GGUF for local use with Ollama.

In [ ]:
# Option A: save LoRA adapters (small, upload to Hugging Face Hub for free hosting)
model.save_pretrained("german_tutor_lora")
tokenizer.save_pretrained("german_tutor_lora")

# Uncomment to push to Hugging Face Hub (free) -- requires huggingface-cli login
# model.push_to_hub("your-username/german-tutor-lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your-username/german-tutor-lora", token="YOUR_HF_TOKEN")

In [ ]:
# Option B: export merged model as GGUF for local use with Ollama (free, runs on your own machine)
model.save_pretrained_gguf("german_tutor_gguf", tokenizer, quantization_method="q4_k_m")

# Then download german_tutor_gguf/*.gguf from the Colab file browser,
# and load it with Ollama using a simple Modelfile (see README.md).

In [ ]:
# Quick test generation
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "Start"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))